<a href="https://colab.research.google.com/github/Kalrfou/Special_Topics2/blob/main/Text_Gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U spacy -q

In [ ]:
import requests

In [ ]:
urls = [
    "https://www.gutenberg.org/files/1342/1342-0.txt",  # Pride & Prejudice
    #"https://www.gutenberg.org/files/11/11-0.txt",      # Alice in Wonderland
   # "https://www.gutenberg.org/files/1661/1661-0.txt"   # Sherlock Holmes
]

all_text = ""

for url in urls:
    all_text += requests.get(url).text

In [ ]:
import numpy as np
import spacy

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.utils import to_categorical

import numpy as np
import spacy
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 1467279

def separate_punc(doc_text):
    result = []

    for token in nlp(doc_text):
        cleaned = token.text.strip().lower()

        if token.is_punct:
            continue
        if token.is_space:
            continue
        if cleaned == "":
            continue

        result.append(cleaned)

    return result

# Read file
#with open("/content/book1.txt", "r", encoding="utf-8") as f:
    #text = f.read()

# Tokenize
tokens = separate_punc(all_text)

print("Total tokens:", len(tokens))
print("First 20 tokens:", tokens[:20])

# Build fixed-length sequences
train_len = 26
text_sequences = []

for i in range(train_len, len(tokens)):
    seq = tokens[i-train_len:i]
    text_sequences.append(seq)

print("Total sequences:", len(text_sequences))
print("Sample token sequence:", text_sequences[0])

# Convert each token sequence to clean text
clean_sequences = [" ".join(seq) for seq in text_sequences]

# Keras tokenization
tokenizer = Tokenizer(filters='', lower=True)
tokenizer.fit_on_texts(clean_sequences)
sequences = tokenizer.texts_to_sequences(clean_sequences)

vocabulary_size = len(tokenizer.word_counts) + 1
print("Vocabulary size:", vocabulary_size)

# Check lengths before padding
lengths = set(len(seq) for seq in sequences)
print("Unique sequence lengths before padding:", lengths)

# Force same length
sequences = pad_sequences(sequences, maxlen=train_len, padding='pre', truncating='pre')

print("Sequences shape:", sequences.shape)
print("First encoded sequence:", sequences[0])



Total tokens: 128709
First 20 tokens: ['start', 'of', 'the', 'project', 'gutenberg', 'ebook', '1342', 'illustration', 'george', 'allen', 'publisher', '156', 'charing', 'cross', 'road', 'london', 'ruskin', 'house', 'illustration', 'reading']
Total sequences: 128683
Sample token sequence: ['start', 'of', 'the', 'project', 'gutenberg', 'ebook', '1342', 'illustration', 'george', 'allen', 'publisher', '156', 'charing', 'cross', 'road', 'london', 'ruskin', 'house', 'illustration', 'reading', 'jane', '’s', 'letters', 'chap', '34', 'pride']
Vocabulary size: 6990
Unique sequence lengths before padding: {26}
Sequences shape: (128683, 26)
First encoded sequence: [3131    3    1 4145 6987 6988 6989  114  342  424 6986 3130 4141 1854
  869  295 4137  151  114  910   70   31  831 6984 6983  302]


In [ ]:
# =========================
# 4. Organize tokens into sequences
# =========================
train_len = 25 + 1   # 25 input words + 1 target word
text_sequences = []

for i in range(train_len, len(tokens)):
    seq = tokens[i - train_len:i]
    text_sequences.append(" ".join(seq))

print("Total sequences:", len(text_sequences))
print("Sample sequence:", text_sequences[0])

# =========================
# 5. Tokenization
# =========================
tokenizer = Tokenizer(filters='', lower=True)
tokenizer.fit_on_texts(text_sequences)
sequences = tokenizer.texts_to_sequences(text_sequences)

vocabulary_size = len(tokenizer.word_counts) + 1
print("Vocabulary size:", vocabulary_size)

# Check all sequence lengths
lengths = set(len(seq) for seq in sequences)
print("Unique sequence lengths:", lengths)

# Convert to NumPy array
sequences = np.array(sequences)

print("Sequences shape:", sequences.shape)
print("First encoded sequence:", sequences[0])

# =========================
# 6. Prepare X and y
# =========================
X = sequences[:, :-1]
y = sequences[:, -1]

y = to_categorical(y, num_classes=vocabulary_size)

print("X shape:", X.shape)
print("y shape:", y.shape)


Total sequences: 128683
Sample sequence: start of the project gutenberg ebook 1342 illustration george allen publisher 156 charing cross road london ruskin house illustration reading jane ’s letters chap 34 pride
Vocabulary size: 6990
Unique sequence lengths: {26}
Sequences shape: (128683, 26)
First encoded sequence: [3131    3    1 4145 6987 6988 6989  114  342  424 6986 3130 4141 1854
  869  295 4137  151  114  910   70   31  831 6984 6983  302]
X shape: (128683, 25)
y shape: (128683, 6990)


In [ ]:
# =========================
# 7. Create model
# =========================
def create_model(vocabulary_size, seq_len):
    model = Sequential()
    model.add(Embedding(input_dim=vocabulary_size, output_dim=15, input_length=seq_len))
    model.add(LSTM(150, return_sequences=True))
    model.add(LSTM(150))
    model.add(Dense(150, activation='relu'))
    model.add(Dense(vocabulary_size, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

model = create_model(vocabulary_size, X.shape[1])
model.summary()

# =========================
# 8. Train model
# =========================
model.fit(X, y, batch_size=256, epochs=200, verbose=1)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 183s 355ms/step - accuracy: 0.0415 - loss: 6.4269
Epoch 2/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 198s 347ms/step - accuracy: 0.0631 - loss: 5.9966
Epoch 3/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 176s 350ms/step - accuracy: 0.0762 - loss: 5.8193
Epoch 4/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 178s 355ms/step - accuracy: 0.0789 - loss: 5.7168
Epoch 5/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 177s 351ms/step - accuracy: 0.0866 - loss: 5.5923
Epoch 6/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 176s 349ms/step - accuracy: 0.0988 - loss: 5.4644
Epoch 7/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 204s 354ms/step - accuracy: 0.1104 - loss: 5.3311
Epoch 8/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 203s 355ms/step - accuracy: 0.1184 - loss: 5.2129
Epoch 9/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 198s 347ms/step - accuracy: 0.1231 - loss: 5.1074
Epoch 10/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 177s 351ms/step - accuracy: 0.1285 - loss: 5.0157
Epoch 11/200
503/503 ━━━━━━━━━━━━━━━━━━━━ 176s 350ms/step - accuracy: 0.1323 - loss: 4.92

In [ ]:
# =========================
# 9. Text generation function
# =========================
def generate_text(model, tokenizer, seq_len, seed_text, num_gen_words):
    output_text = []
    input_text = seed_text

    for _ in range(num_gen_words):
        encoded_text = tokenizer.texts_to_sequences([input_text])[0]
        pad_encoded = pad_sequences([encoded_text], maxlen=seq_len, truncating='pre')

        predict_x = model.predict(pad_encoded, verbose=0)[0]
        pred_word_ind = np.argmax(predict_x)

        pred_word = tokenizer.index_word.get(pred_word_ind, "")

        if pred_word == "":
            break

        input_text += " " + pred_word
        output_text.append(pred_word)

    return " ".join(output_text)

# =========================
# 10. Test generation
# =========================
seed_text = "the sky is"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 50)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " "+ generated)
print("==================================================================")
# =========================
# 11. Save model
# =========================
model.save("lstm_text_generator.h5")
print("Model saved as lstm_text_generator.h5")


Seed text: the sky is
Generated text: the sky is their whispering to the pride of the regiment ’s remaining on the country or for universal and the two gentlemen were spent pursuing time paid dining consolation of the valley it soon and behind by the face and as she had something to find that led that they had any
Model saved as lstm_text_generator.h5


In [ ]:
seed_text = "What the next"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 50)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " "+ generated)
print("==================================================================")


Seed text: What the next
Generated text: What the next embarrassment which connects of joy and the conclusion of her being proud that he expressed no end to rosings and whenever she came to the window between the room where she was tempted by the pleasantness of the morning and pack they vain the example she assured him to point


In [ ]:
seed_text = "another exampl for this topic"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 15)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " "+ generated)
print("==================================================================")


Seed text: another exampl for this topic
Generated text: another exampl for this topic could be more than she has rather given to ordinary occasions i could not be


In [ ]:
seed_text = "where did she go?"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 15)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " "+ generated)
print("==================================================================")


Seed text: where did she go?
Generated text: where did she go? loved him to come with the church yours i think to obtain impossible to be
